In [ ]:
import pandas as pd

In [ ]:
raw_training_data = pd.read_csv('../data/train.csv')

In [ ]:
raw_training_data.dtypes

In [ ]:
from sklearn.preprocessing import OneHotEncoder


In [ ]:
categorical_features = raw_training_data.select_dtypes(include='string').columns
categorical_features

In [ ]:
for col in categorical_features:
    print(f"\nVariable: {col}")
    print(f"Unique categories: {raw_training_data[col].nunique()}")
    print(raw_training_data[col].value_counts())
    print("-" * 40)

In [ ]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=int
)

X_encoded = encoder.fit_transform(raw_training_data[categorical_features])

In [ ]:
encoder.categories_

In [ ]:
for col, category in zip(categorical_features, encoder.categories_):
    print(f"{col}: {category}\n")

In [ ]:
encoder.get_feature_names_out(categorical_features)

In [ ]:
encoded_df = pd.DataFrame(
    X_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=raw_training_data.index
)

encoded_df.head()

In [ ]:
with pd.option_context('display.max_columns', None):
    print(encoded_df.head())

In [ ]:
encoded_df.columns

In [ ]:
raw_training_data.columns

In [ ]:
raw_training_data_numeric = raw_training_data.drop(columns=categorical_features)
raw_training_data_numeric.head()

In [ ]:
processed_data = pd.concat([raw_training_data_numeric, encoded_df], axis=1)

print("Original shape:", raw_training_data.shape)
print("Processed shape:", processed_data.shape)
print("Missing values:", processed_data.isnull().sum().sum())

In [ ]:
processed_data.columns
train_X = processed_data.drop(columns=["is_fraud"])
train_y = processed_data["is_fraud"]

In [ ]:
processed_data.dtypes

In [ ]:
# might want to comeback and scale the numeric features

In [ ]:
# probably move to another note book but logistic regression modeling below

In [ ]:
# actually before i do I need to apply the preprocessing steps to my validation data

In [ ]:
testing_data = pd.read_csv("../data/test.csv")
testing_data.shape
raw_training_data.shape

In [ ]:
testing_data.columns

In [ ]:
# so the testing data is for the kaggle testing. I thought they already split the testing data. I need to go back and clean up the code.

In [1]:
# I will just restart here with what i know now
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [2]:
training_data = pd.read_csv("../data/train.csv")
training_data.shape

(699960, 26)

In [3]:
X = training_data.drop(columns=["is_fraud"])
y = training_data["is_fraud"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=63,
    stratify=y
)
# there wasnt a data library or background on the data. I think its reasonable to assume each ID represents a transaction. In the EDA, I noticed the target variable was largely imbalanced. Thus i statified on y.

In [4]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 559968 entries, 579956 to 386042
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            559968 non-null  int64  
 1   amount_usd                    559968 non-null  float64
 2   hours_since_last_txn          559968 non-null  float64
 3   txn_count_last_24h            559968 non-null  int64  
 4   distance_from_home_km         559968 non-null  float64
 5   card_age_months               559968 non-null  int64  
 6   customer_age                  559968 non-null  int64  
 7   account_balance_usd           559968 non-null  float64
 8   cvv_retry_count               559968 non-null  int64  
 9   velocity_score                559968 non-null  float64
 10  time_of_day_hour              559968 non-null  int64  
 11  day_of_week                   559968 non-null  int64  
 12  merchant_risk_score           559968 non-null  float64


In [5]:
y_train.info()

<class 'pandas.Series'>
Index: 559968 entries, 579956 to 386042
Series name: is_fraud
Non-Null Count   Dtype  
--------------   -----  
559968 non-null  float64
dtypes: float64(1)
memory usage: 8.5 MB


In [6]:
y_train.value_counts()

is_fraud
0.0    550617
1.0      9351
Name: count, dtype: int64

In [8]:
categorical_features = X_train.select_dtypes(include=["string"]).columns
print(len(categorical_features))

5


In [9]:
for col in categorical_features:
    print(f"\nVariable: {col}")
    print(f"Unique categories: {training_data[col].nunique()}")
    print(training_data[col].value_counts())
    print("-" * 40)


Variable: merchant_category
Unique categories: 12
merchant_category
Online Retail      155097
Groceries          142578
Restaurants        101360
Fuel                59880
Electronics         48693
Streaming           47196
Utilities           42027
Healthcare          35889
Travel              28715
Gaming              23784
Gift Cards           9456
Crypto Exchange      5285
Name: count, dtype: int64
----------------------------------------

Variable: card_type
Unique categories: 5
card_type
Visa          345001
Mastercard    254857
Amex           45797
RuPay          37070
Discover       17235
Name: count, dtype: int64
----------------------------------------

Variable: auth_method
Unique categories: 5
auth_method
3D Secure            210036
OTP                  175178
PIN                  154020
Biometric             93666
No Authentication     67060
Name: count, dtype: int64
----------------------------------------

Variable: channel
Unique categories: 5
channel
Online         27

In [11]:
expected_dependent_features_count = 25-5+8+5+5+5+12

In [12]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=int
)

X_train_encoded = encoder.fit_transform(X_train[categorical_features])
X_val_encoded = encoder.transform(X_val[categorical_features])

In [14]:
print(X_train_encoded.shape)
print(X_val_encoded.shape)

(559968, 35)
(139992, 35)


In [ ]:
encoded_train_df = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_train.index
)

X_train_numeric = X_train.drop(columns=categorical_features)
X_train_processed = pd.concat([X_train_numeric, encoded_train_df], axis=1)
print("Processed training shape:", X_train_processed.shape)

Processed shape: (559968, 55)


In [17]:
encoded_val_df = pd.DataFrame(
    X_val_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_val.index
)

X_val_numeric = X_val.drop(columns=categorical_features)
X_val_processed = pd.concat([X_val_numeric, encoded_val_df], axis=1)
print("Processed validation shape:", X_train_processed.shape)

Processed validation shape: (559968, 55)


In [23]:
assert expected_dependent_features_count == X_train_processed.shape[1] == X_val_processed.shape[1]

In [27]:
print(y_train.dtype, y_val.dtype)

float64 float64


In [28]:
print(y_train.unique())
print(y_val.unique())


[0. 1.]
[0. 1.]


In [29]:
print(y_train.value_counts(normalize=True) * 100)
print(y_val.value_counts(normalize=True) *100)

is_fraud
0.0    98.330083
1.0     1.669917
Name: proportion, dtype: float64
is_fraud
0.0    98.329905
1.0     1.670095
Name: proportion, dtype: float64


In [32]:
print(X_train_processed.index.equals(y_train.index))
print(X_val_processed.index.equals(y_val.index))
print(X_train_processed.columns.equals(X_val_processed.columns))

True
True
True


# Moving on to the modeling

In [ ]:
# train and evaluate the logistic regression model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_)